# Overview

This notebook corresponds to makemore part 3.


In [ ]:
import random

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Load Data and Build Datasets

Removed checks and EDA (covered in part 2 notebook)


In [ ]:
with open("names.txt", "r") as file:
    names = file.read().splitlines()

In [ ]:
chars = ["."] + sorted(set("".join(names)))
s_to_i = {s: i for i, s in enumerate(chars)}
i_to_s = {i: s for i, s in enumerate(chars)}

vocab_size = len(chars)


def build_dataset(names, block_size=3):
    X, y = [], []
    for name in names:
        context = [0] * block_size
        name += "."
        for ch in name:
            idx = s_to_i[ch]
            X.append(context)
            y.append(idx)
            context = context[1:] + [idx]

    return torch.tensor(X), torch.tensor(y)

In [ ]:
random.seed(42)
random.shuffle(names)

p1, p2 = int(len(names) * 0.8), int(len(names) * 0.9)

block_size = 3

X_train, y_train = build_dataset(names[:p1], block_size)
X_val, y_val = build_dataset(names[p1:p2], block_size)
X_test, y_test = build_dataset(names[p2:], block_size)

# Build MLP


In [ ]:
g = torch.Generator().manual_seed(2147483647)

embedding_dim = 10
hidden_size = 200

eps = 1e-5
bn_momentum = 1e-3

C = torch.randn((vocab_size, embedding_dim), generator=g)

# Use Kaiming initialization (tanh version)
W1 = torch.randn((embedding_dim * block_size, hidden_size), generator=g) * (
    (5 / 3) / (embedding_dim * block_size) ** 0.5
)
# Omit bias since using BN

bn_running_mean = torch.zeros((1, hidden_size))
bn_running_var = torch.ones((1, hidden_size))
bn_gain = torch.ones((1, hidden_size))
bn_bias = torch.zeros((1, hidden_size))

# Scale down W2 and b2 to prevent extreme logits
W2 = torch.randn((hidden_size, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0

# params = [C, W1, b1, W2, b2]
params = [C, W1, bn_gain, bn_bias, W2, b2]
for p in params:
    p.requires_grad = True

print(f"Total parameters: {sum(p.numel() for p in params)}")

# Train and Evaluate MLP


In [ ]:
batch_size = 256
num_steps = 200000
lrs = [0.02, 0.2, 0.02, 0.002]
milestones = [2000, 120000, 150000]

losses = []
j = 0
for i in range(num_steps):
    # Get mini-batch
    idxs = torch.randint(0, X_train.shape[0], (batch_size,), generator=g)
    X_batch, y_batch = X_train[idxs], y_train[idxs]

    # Forward pass

    # Embedding layer
    embs = C[X_batch]
    emb_concat = embs.reshape((len(X_batch), -1))

    # Linear layer
    preact = emb_concat @ W1

    # BN layer
    batch_mean = preact.mean(dim=0, keepdims=True)
    batch_var = preact.var(dim=0, keepdims=True)
    bn_preact = bn_gain * (preact - batch_mean) / torch.sqrt(batch_var + eps) + bn_bias

    # Update BN running stats
    with torch.no_grad():
        # PyTorch's momentum def'n is opposite of standard def'n for historical reasons
        bn_running_mean = (1 - bn_momentum) * bn_running_mean + bn_momentum * batch_mean
        bn_running_var = (1 - bn_momentum) * bn_running_var + bn_momentum * batch_var

    # Activation layer
    h = torch.tanh(bn_preact)

    # Output layer
    logits = h @ W2 + b2

    # Compute loss

    loss = F.cross_entropy(logits, y_batch)

    # Backward pass

    # Backprop
    for p in params:
        p.grad = None
    loss.backward()

    # Set learning rate
    if j < len(milestones) and i >= milestones[j]:
        j += 1
    lr = lrs[j]

    # GD step
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad

    # Track loss
    losses.append(loss.item())

    if i % 10000 == 0:
        print(f"Epoch: {i}/{num_steps}, loss: {loss}")

Confirm that BN running statistics are reasonable


In [ ]:
embs = C[X_train]
emb_concat = embs.reshape(len(X_train), -1)
preact = emb_concat @ W1
mean = preact.mean(dim=0)
var = preact.var(dim=0)

print(f"Full mean: {mean[:10]}")
print(f"Full var: {var[:10]}")
print(f"Running mean: {bn_running_mean[0, :10]}")
print(f"Running var: {bn_running_var[0, :10]}")

In [ ]:
plt.plot(losses)

In [ ]:
def evaluate(params, X, y):
    C, W1, bn_gain, bn_bias, W2, b2 = params

    embs = C[X]
    emb_concat = embs.reshape((len(X), -1))

    preact = (
        bn_gain * (emb_concat @ W1 - bn_running_mean) / torch.sqrt(bn_running_var + eps)
        + bn_bias
    )
    h = torch.tanh(preact)

    logits = h @ W2 + b2

    return F.cross_entropy(logits, y).item()

In [ ]:
train_loss = evaluate(params, X_train, y_train)
val_loss = evaluate(params, X_val, y_val)

print(f"Train loss: {train_loss}")
print(f"Val loss: {val_loss}")

Val loss:

- Without BN: 2.09651327
- With BN: 2.08763897


# Inference


Extension: BN folding


In [ ]:
W1_fold = (bn_gain * W1) / torch.sqrt(bn_running_var + eps)
bias_fold = -(bn_gain * bn_running_mean) / torch.sqrt(bn_running_var + eps) + bn_bias

In [ ]:
for _ in range(5):
    x = [0] * block_size
    out = ""
    while True:
        embs = C[torch.tensor(x)]
        emb_concat = embs.reshape(1, -1)

        preact = emb_concat @ W1_fold + bias_fold
        h = torch.tanh(preact)

        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)

        idx = torch.multinomial(probs, num_samples=1).item()

        if idx == 0:
            break

        x = x[1:] + [idx]
        out += i_to_s[idx]

    print(out)

Confirm BN folding is correct


In [ ]:
x = [0] * block_size
out_no_fold = ""
sample_g = torch.Generator().manual_seed(42)
while True:
    embs = C[torch.tensor(x)]
    emb_concat = embs.reshape(1, -1)

    preact = (
        bn_gain * (emb_concat @ W1 - bn_running_mean) / torch.sqrt(bn_running_var + eps)
        + bn_bias
    )
    h = torch.tanh(preact)

    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=1)

    idx = torch.multinomial(probs, num_samples=1, generator=sample_g).item()

    if idx == 0:
        break

    x = x[1:] + [idx]
    out_no_fold += i_to_s[idx]

print(f"Output (no BN folding): {out_no_fold}")

x = [0] * block_size
out_fold = ""
sample_g = torch.Generator().manual_seed(42)
while True:
    embs = C[torch.tensor(x)]
    emb_concat = embs.reshape(1, -1)

    preact = emb_concat @ W1_fold + bias_fold
    h = torch.tanh(preact)

    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=1)

    idx = torch.multinomial(probs, num_samples=1, generator=sample_g).item()

    if idx == 0:
        break

    x = x[1:] + [idx]
    out_fold += i_to_s[idx]

print(f"Output (BN folding): {out_fold}")